# Explore here

In [2]:
# Cargamos los datos del censo que nos facilita 4Geeks
import pandas as pd

url = "https://breathecode.herokuapp.com/asset/internal-link?id=2326&path=adult-census-income.csv"

df = pd.read_csv(url)

# Vemos las primeras filas para conocer los datos
df.head(20)

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K
5,34,Private,216864,HS-grad,9,Divorced,Other-service,Unmarried,White,Female,0,3770,45,United-States,<=50K
6,38,Private,150601,10th,6,Separated,Adm-clerical,Unmarried,White,Male,0,3770,40,United-States,<=50K
7,74,State-gov,88638,Doctorate,16,Never-married,Prof-specialty,Other-relative,White,Female,0,3683,20,United-States,>50K
8,68,Federal-gov,422013,HS-grad,9,Divorced,Prof-specialty,Not-in-family,White,Female,0,3683,40,United-States,<=50K
9,41,Private,70037,Some-college,10,Never-married,Craft-repair,Unmarried,White,Male,0,3004,60,?,>50K


In [3]:
# Revisamos la información de nuestro dataset
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             32561 non-null  int64
 1   workclass       32561 non-null  str  
 2   fnlwgt          32561 non-null  int64
 3   education       32561 non-null  str  
 4   education.num   32561 non-null  int64
 5   marital.status  32561 non-null  str  
 6   occupation      32561 non-null  str  
 7   relationship    32561 non-null  str  
 8   race            32561 non-null  str  
 9   sex             32561 non-null  str  
 10  capital.gain    32561 non-null  int64
 11  capital.loss    32561 non-null  int64
 12  hours.per.week  32561 non-null  int64
 13  native.country  32561 non-null  str  
 14  income          32561 non-null  str  
dtypes: int64(6), str(9)
memory usage: 3.7 MB


In [4]:
# Comprobamos si hay datos desconocidos en las columnas
valores_desconocidos = df.astype(str).apply(
    lambda columna: columna.str.strip().eq("?").sum()
)

# Mostramos dónde aparecen esos valores
valores_desconocidos[valores_desconocidos > 0]

workclass         1836
occupation        1843
native.country     583
dtype: int64

In [5]:
# Quitamos los espacios que sobran en las columnas de texto
columnas_texto = df.select_dtypes(include="object").columns

df[columnas_texto] = df[columnas_texto].apply(
    lambda columna: columna.str.strip()
)

# Convertimos los signos ? en valores nulos
df = df.replace("?", pd.NA)

# Eliminamos las filas que tienen información incompleta
df = df.dropna().copy()

# Comprobamos cuántas personas nos quedan
print("Personas después de la limpieza:", len(df))

Personas después de la limpieza: 30162


/tmp/ipykernel_3633/367696096.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  columnas_texto = df.select_dtypes(include="object").columns


In [6]:
# Vemos cómo se reparten los ingresos de las personas
df["income"].value_counts()

income
<=50K    22654
>50K      7508
Name: count, dtype: int64

In [7]:
# Separamos los datos de la respuesta que queremos predecir
X = df.drop(columns=["income"])
y = df["income"].map({"<=50K": 0, ">50K": 1})

# Comprobamos que las dos categorías están bien preparadas
y.value_counts()

income
0    22654
1     7508
Name: count, dtype: int64

In [8]:
# Separamos los datos para entrenar y probar nuestro modelo
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Comprobamos cuántas personas hay en cada grupo
print("Entrenamiento:", len(X_train))
print("Prueba:", len(X_test))

Entrenamiento: 24129
Prueba: 6033


In [9]:
# Preparamos los datos para nuestro modelo
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Separamos las columnas numéricas de las categóricas
columnas_numericas = X_train.select_dtypes(include="number").columns
columnas_categoricas = X_train.columns.difference(columnas_numericas)

# Normalizamos los números y convertimos los textos en números
preprocesador = ColumnTransformer([
    ("numericas", StandardScaler(), columnas_numericas),
    ("categoricas", OneHotEncoder(handle_unknown="ignore"), columnas_categoricas)
])

# Preparamos los datos de entrenamiento y prueba
X_train_preparado = preprocesador.fit_transform(X_train)
X_test_preparado = preprocesador.transform(X_test)

print("Entrenamiento:", X_train_preparado.shape)
print("Prueba:", X_test_preparado.shape)


Entrenamiento: (24129, 103)
Prueba: (6033, 103)


Queremos crear un sistema que ayude a una persona a explorar posibles opciones para mejorar sus oportunidades de ingresos.

Cada usuario será una persona del dataset, con sus estudios, profesión, edad y horas de trabajo.

Buscaremos perfiles parecidos y compararemos sus trayectorias para recomendar posibles opciones de formación o empleo.

Estas recomendaciones se basarán en los datos, pero no garantizan que una persona vaya a ganar más dinero.

In [10]:
# Elegimos los datos que utilizaremos para recomendar
variables = ["education", "occupation", "workclass", "hours.per.week"]

from sklearn.neighbors import NearestNeighbors

# Preparamos los perfiles para poder compararlos
preparador = ColumnTransformer([
    ("numeros", StandardScaler(), ["hours.per.week"]),
    ("texto", OneHotEncoder(handle_unknown="ignore"),
     ["education", "occupation", "workclass"])
])

perfiles = preparador.fit_transform(X_train[variables])

# Nos quedamos con los perfiles que ganan más de 50.000 dólares
perfiles_altos = perfiles[y_train.to_numpy() == 1]

# Preparamos el sistema para buscar personas similares
recomendador = NearestNeighbors(n_neighbors=5, metric="cosine")
recomendador.fit(perfiles_altos)

print("Perfiles de referencia:", perfiles_altos.shape[0])

Perfiles de referencia: 6006


In [11]:
# Creamos una persona ficticia para probar el recomendador
perfil_usuario = pd.DataFrame([{
    "education": "HS-grad",
    "occupation": "Sales",
    "workclass": "Private",
    "hours.per.week": 20
}])

# Preparamos sus datos y buscamos perfiles parecidos
perfil_preparado = preparador.transform(perfil_usuario)

distancias, posiciones = recomendador.kneighbors(perfil_preparado)

# Recuperamos los cinco perfiles encontrados
personas_ingresos_altos = X_train.loc[
    y_train == 1, variables
].reset_index(drop=True)

recomendaciones = personas_ingresos_altos.iloc[posiciones[0]]

recomendaciones

,education,occupation,workclass,hours.per.week
4866,HS-grad,Sales,Private,26
4022,HS-grad,Sales,Private,30
15,HS-grad,Sales,Private,35
1017,HS-grad,Sales,Self-emp-inc,10
2310,HS-grad,Prof-specialty,Private,10


In [12]:
# Probamos el recomendador con otra persona ficticia
otro_usuario = pd.DataFrame([{
    "education": "Some-college",
    "occupation": "Adm-clerical",
    "workclass": "Private",
    "hours.per.week": 40
}])

# Buscamos perfiles parecidos con ingresos altos
otro_perfil = preparador.transform(otro_usuario)

distancias, posiciones = recomendador.kneighbors(otro_perfil)

# Vemos los resultados
personas_ingresos_altos.iloc[posiciones[0]]

,education,occupation,workclass,hours.per.week
1079,Some-college,Adm-clerical,Private,40
5935,Some-college,Adm-clerical,Private,40
1134,Some-college,Adm-clerical,Private,40
4068,Some-college,Adm-clerical,Private,40
771,Some-college,Adm-clerical,Private,40


In [13]:
# Entrenamos un modelo para predecir los ingresos
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

modelo = LogisticRegression(max_iter=1000)

# Entrenamos con los datos preparados
modelo.fit(X_train_preparado, y_train)

# Predecimos los ingresos de las personas de prueba
predicciones = modelo.predict(X_test_preparado)

# Vemos qué resultado consigue
print("Accuracy:", accuracy_score(y_test, predicciones))
print(classification_report(y_test, predicciones))

Accuracy: 0.8543013426156141
              precision    recall  f1-score   support

           0       0.88      0.93      0.91      4531
           1       0.75      0.62      0.68      1502

    accuracy                           0.85      6033
   macro avg       0.82      0.78      0.79      6033
weighted avg       0.85      0.85      0.85      6033



In [14]:
# Probamos XGBoost para ver si mejora nuestro resultado
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

# Creamos y entrenamos el modelo
modelo_xgb = XGBClassifier(
    n_estimators=100,
    random_state=42
)

modelo_xgb.fit(X_train_preparado, y_train)

# Hacemos las predicciones
predicciones_xgb = modelo_xgb.predict(X_test_preparado)

# Comparamos los resultados
print("Regresión logística: 85,43 %")
print("XGBoost:", round(accuracy_score(y_test, predicciones_xgb) * 100, 2), "%")

print(classification_report(y_test, predicciones_xgb))

Regresión logística: 85,43 %
XGBoost: 87.67 %
              precision    recall  f1-score   support

           0       0.90      0.94      0.92      4531
           1       0.80      0.68      0.73      1502

    accuracy                           0.88      6033
   macro avg       0.85      0.81      0.83      6033
weighted avg       0.87      0.88      0.87      6033



### Conclusión del proyecto

En este proyecto hemos trabajado con datos del censo de Estados Unidos para analizar los ingresos de las personas y crear un sistema de recomendaciones.

Después de limpiar y preparar los datos, entrenamos dos modelos para predecir si una persona gana más de 50.000 dólares al año.

Con regresión logística conseguimos un **85,43 % de accuracy**, y después probamos XGBoost, que mejoró el resultado hasta el **87,67 %**.

También construimos un recomendador con KNN y lo probamos con dos personas ficticias. El sistema nos permitió encontrar perfiles parecidos con ingresos altos y comparar sus estudios, profesiones y horas de trabajo.

Estas recomendaciones pueden servir para explorar otras opciones laborales, aunque no garantizan que esos cambios aumenten los ingresos.

En general, hemos conseguido construir y probar el sistema que nos pedía el proyecto.